In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl
!pip install transformers

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-51iuj5t9/unsloth_e4086db0082b4dc99d5d6eca434f29d1
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-51iuj5t9/unsloth_e4086db0082b4dc99d5d6eca434f29d1
  Resolved https://github.com/unslothai/unsloth.git to commit 93a24f66980b2820083d2882fd3a720d894a1414
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.4/418.4 kB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 132.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 23.6 MB/s eta 0:00:0

In [2]:

import torch
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel

max_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    max_seq_length=max_length,
    dtype=None,
    load_in_4bit=True,
)

/tmp/ipykernel_4860/3702060121.py:3: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


model.safetensors:   0%|          | 0.00/1.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = True,
    random_state = 3407,
)

In [4]:
from datasets import load_dataset

dataset = load_dataset(
    "HuggingFaceTB/Countdown-Task-GOLD",
    "verified_Qwen2.5-7B-Instruct",
    split="train[:10000]"
)

def format_prompts(batch):
    prompts_list = batch["messages"]
    texts = []

    for messages in prompts_list:
        if len(messages) >= 3:
            user_text = messages[1]['content']
            assistant_text = messages[2]['content']
            text = f"<start_of_turn>user\n{user_text}<end_of_turn>\n<start_of_turn>model\n{assistant_text}<end_of_turn>"
        else:
            text = ""
        texts.append(text)

    return {"text": texts}


dataset = dataset.map(format_prompts, batched = True)

print(dataset[0]["text"])

README.md: 0.00B [00:00, ?B/s]

verified_Qwen2.5-7B-Instruct/train-00000(…):   0%|          | 0.00/11.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/30441 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

<start_of_turn>user
Using the numbers [78, 46, 93], create an equation that equals 61. You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. Show your work in <think> </think> tags. And return the final equation and answer in <answer> </answer> tags, for example <answer> (1 + 2) / 3 = 1 </answer>.<end_of_turn>
<start_of_turn>model
<think>
To solve this, I need to use the numbers 78, 46, and 93 to create an equation that equals 61. I'll start by considering the basic arithmetic operations and how they can be used to get close to 61.

First, I'll try subtraction, as it's the most straightforward operation to get close to 61:
- 78 - 46 = 32 (too low)
- 93 - 46 = 47 (closer but still too low)
- 93 - 78 = 15 (too low)

Next, I'll try addition and subtraction in combination:
- 78 - 93 = -15 (too low, and I need to add something positive)
- 46 + 93 = 139 (too high, and I need to subtract something)

Since 93 is the highest number, I'll try to use it in a 

In [5]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=2048,
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    num_proc=1,
    remove_columns=dataset.column_names,
)

Map (num_proc=1):   0%|          | 0/10000 [00:00<?, ? examples/s]

In [6]:
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForLanguageModeling
from unsloth import is_bfloat16_supported

sft_config = SFTConfig(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 10,
    max_steps = 1000,
    learning_rate = 2e-4,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs",
    push_to_hub = False,
    report_to = "none",
)

trainer = SFTTrainer(
    model = model,
    train_dataset = tokenized_dataset,
    args = sft_config,
    processing_class = tokenizer,
    data_collator = DataCollatorForLanguageModeling(tokenizer = tokenizer, mlm = False),
)

trainer_stats = trainer.train()

Unsloth: Switching to float32 training since model cannot work with float16


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 1 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 13,045,760 of 1,012,931,712 (1.29% trained)


Step,Training Loss
1,1.994488
2,2.290506
3,1.853840
4,1.528903
5,1.630344
6,1.259167
7,1.268317
8,1.043942
9,1.022727
10,0.927661


In [7]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

('lora_model/tokenizer_config.json',
 'lora_model/chat_template.jinja',
 'lora_model/tokenizer.json')

In [8]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_model",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

prompt = "<prompt>Используя числа [8, 8, 3, 3], получи 24</prompt>\n<think>"
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

print("Начинаю генерацию... (это может занять до минуты)")

outputs = model.generate(
    **inputs,
    max_new_tokens = 512,
    repetition_penalty = 1.2,
    no_repeat_ngram_size = 3,
    temperature = 0.4,
    top_p = 0.9,
    use_cache = True,
    pad_token_id = tokenizer.eos_token_id
)

print("-" * 30)
print(tokenizer.decode(outputs[0], skip_special_tokens = True))

==((====))==  Unsloth 2026.4.4: Fast Gemma3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Начинаю генерацию... (это может занять до минуты)
------------------------------
<prompt>Используя числа [8, 8, 3, 3], получи 24</prompt>
<think>

To solve this problem. I need to find a combination of the numbers 79 and 10 that will give me 65 using basic arithmetic operations (addition, subtraction), but since we can't use 1 = 3 or 3 + 3 in any way because they would be too large). Let's try division and multiplication with 8 as it seems likely to get close to 20.

If we divide 8 by 8:
- 8 / 8= 1 , which is not helpful.

Let's consider 3 - 3*2 = 1 . This doesn't work either. However, if we multiply 8 * 3 = 26, then adding 1 to 16 gives us 62. We still have 3 left, so let's see how we can make 22 from them. If we subtract 3 from 28, we get 23. So, 27 may help. But we don't have 2 како as our number.**">

The correct approach might be to add 8 and 3 to get 11, and then subtracting 3 helps us reach 64.. Now, we need to adjust to get to 68. The only way for us to do this without creating

In [ ]:
import pandas as pd
import re
from tqdm import tqdm
from unsloth import FastLanguageModel

test_df = pd.read_csv("test_public.csv")
results = []

FastLanguageModel.for_inference(model)

print("Начинаю генерацию ответов для 2000 задач...")

for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    target = row['target']
    nums = row['nums']

    prompt = f"<prompt>Используя числа {nums}, получи {target}</prompt>\n<think>"

    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens = 1024,
        repetition_penalty = 1.2,
        temperature = 0.1,
        use_cache = True
    )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens = True)

    match = re.search(r'<answer>(.*?)(?:</answer>|$)', full_text, re.DOTALL)
    equation = match.group(1).strip() if match else ""

    if not equation:
        equation = full_text.split('\n')[-1].strip()

    results.append({
        "id": row['id'],
        "equation": equation
    })

submission_df = pd.DataFrame(results)
submission_df.to_csv("my_submission.csv", index=False)

print("\nГотово! Файл my_submission.csv создан.")

Начинаю генерацию ответов для 2000 задач...


  0%|          | 1/2000 [05:30<183:21:47, 330.22s/it]Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
